In [39]:
import sys

print(sys.executable)


c:\Documents\Desktop\Fintech_Api-Pipeline\venv\Scripts\python.exe


### Import the Libraries

In [40]:
import os
from datetime import datetime, timezone

import pandas as pd

import requests

from dotenv import load_dotenv

from sqlalchemy import create_engine, text

In [41]:
# load environment variables from .env file
load_dotenv()


True

### Define API endpoints/configuration

In [42]:
BASE_CURRENCY = "USD"
API_URL = (f"https://open.er-api.com/v6/latest/{BASE_CURRENCY}")

print(API_URL)


https://open.er-api.com/v6/latest/USD


### Extraction

In [43]:
response = requests.get(API_URL, timeout=30)

print("status code:", response.status_code)
response.raise_for_status()  # Raise an exception for HTTP errors

api_data = response.json()

print(api_data["result"])
print(api_data["base_code"])

status code: 200
success
USD


In [44]:
# Inspect the complete JSON response to understand its structure
api_data  # This will display the entire JSON response in the notebook

{'result': 'success',
 'provider': 'https://www.exchangerate-api.com',
 'documentation': 'https://www.exchangerate-api.com/docs/free',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1784332952,
 'time_last_update_utc': 'Sat, 18 Jul 2026 00:02:32 +0000',
 'time_next_update_unix': 1784421102,
 'time_next_update_utc': 'Sun, 19 Jul 2026 00:31:42 +0000',
 'time_eol_unix': 0,
 'base_code': 'USD',
 'rates': {'USD': 1,
  'AED': 3.6725,
  'AFN': 66.008291,
  'ALL': 81.852701,
  'AMD': 366.674156,
  'ANG': 1.79,
  'AOA': 926.855963,
  'ARS': 1480.1223,
  'AUD': 1.432241,
  'AWG': 1.79,
  'AZN': 1.699923,
  'BAM': 1.710272,
  'BBD': 2,
  'BDT': 123.245194,
  'BGN': 1.710272,
  'BHD': 0.376,
  'BIF': 2992.642076,
  'BMD': 1,
  'BND': 1.291343,
  'BOB': 10.533892,
  'BRL': 5.1005,
  'BSD': 1,
  'BTN': 96.409744,
  'BWP': 14.045655,
  'BYN': 2.88484,
  'BZD': 2,
  'CAD': 1.401869,
  'CDF': 2272.550711,
  'CHF': 0.807431,
  'CLF': 0.023396,
  'CLP': 924.739337,
 

In [45]:
# Extract the business data out of the main JSON output
api_data["rates"]  # This will display the exchange rates data in the notebook

{'USD': 1,
 'AED': 3.6725,
 'AFN': 66.008291,
 'ALL': 81.852701,
 'AMD': 366.674156,
 'ANG': 1.79,
 'AOA': 926.855963,
 'ARS': 1480.1223,
 'AUD': 1.432241,
 'AWG': 1.79,
 'AZN': 1.699923,
 'BAM': 1.710272,
 'BBD': 2,
 'BDT': 123.245194,
 'BGN': 1.710272,
 'BHD': 0.376,
 'BIF': 2992.642076,
 'BMD': 1,
 'BND': 1.291343,
 'BOB': 10.533892,
 'BRL': 5.1005,
 'BSD': 1,
 'BTN': 96.409744,
 'BWP': 14.045655,
 'BYN': 2.88484,
 'BZD': 2,
 'CAD': 1.401869,
 'CDF': 2272.550711,
 'CHF': 0.807431,
 'CLF': 0.023396,
 'CLP': 924.739337,
 'CNH': 6.777672,
 'CNY': 6.784582,
 'COP': 3225.081918,
 'CRC': 454.018753,
 'CUP': 24,
 'CVE': 96.421037,
 'CZK': 21.157415,
 'DJF': 177.721,
 'DKK': 6.526812,
 'DOP': 58.370612,
 'DZD': 133.061169,
 'EGP': 50.543868,
 'ERN': 15,
 'ETB': 160.889769,
 'EUR': 0.874447,
 'FJD': 2.242714,
 'FKP': 0.743836,
 'FOK': 6.526799,
 'GBP': 0.743836,
 'GEL': 2.631036,
 'GGP': 0.743836,
 'GHS': 11.568737,
 'GIP': 0.743836,
 'GMD': 74.331983,
 'GNF': 8777.194785,
 'GTQ': 7.626168,


In [46]:
type(api_data["rates"])  # This will show the type of the extracted rates data

dict

In [47]:
api_data.keys()  # This will display the keys of the main JSON response

dict_keys(['result', 'provider', 'documentation', 'terms_of_use', 'time_last_update_unix', 'time_last_update_utc', 'time_next_update_unix', 'time_next_update_utc', 'time_eol_unix', 'base_code', 'rates'])

### save the raw API response

In [48]:
import json 
from pathlib import Path

raw_folder = Path("../data/raw_data") 

raw_folder.mkdir(parents=True, exist_ok=True)

extraction_timestamp = datetime.now(timezone.utc)

raw_filename = (
    f"exchange_rates_"
    f"{extraction_timestamp:%Y%m%d_%H%M%S}.json"
) 

raw_path = raw_folder / raw_filename

with open(raw_path, "w", encoding="utf-8") as file:
    json.dump(api_data, file, indent=4) 
    
print(f"Raw response saved to: {raw_path}")

Raw response saved to: ..\data\raw_data\exchange_rates_20260718_202129.json


### Transformation

In [61]:
# Extract the business data 
rates = api_data["rates"]

# Display the data
rates

{'USD': 1,
 'AED': 3.6725,
 'AFN': 66.008291,
 'ALL': 81.852701,
 'AMD': 366.674156,
 'ANG': 1.79,
 'AOA': 926.855963,
 'ARS': 1480.1223,
 'AUD': 1.432241,
 'AWG': 1.79,
 'AZN': 1.699923,
 'BAM': 1.710272,
 'BBD': 2,
 'BDT': 123.245194,
 'BGN': 1.710272,
 'BHD': 0.376,
 'BIF': 2992.642076,
 'BMD': 1,
 'BND': 1.291343,
 'BOB': 10.533892,
 'BRL': 5.1005,
 'BSD': 1,
 'BTN': 96.409744,
 'BWP': 14.045655,
 'BYN': 2.88484,
 'BZD': 2,
 'CAD': 1.401869,
 'CDF': 2272.550711,
 'CHF': 0.807431,
 'CLF': 0.023396,
 'CLP': 924.739337,
 'CNH': 6.777672,
 'CNY': 6.784582,
 'COP': 3225.081918,
 'CRC': 454.018753,
 'CUP': 24,
 'CVE': 96.421037,
 'CZK': 21.157415,
 'DJF': 177.721,
 'DKK': 6.526812,
 'DOP': 58.370612,
 'DZD': 133.061169,
 'EGP': 50.543868,
 'ERN': 15,
 'ETB': 160.889769,
 'EUR': 0.874447,
 'FJD': 2.242714,
 'FKP': 0.743836,
 'FOK': 6.526799,
 'GBP': 0.743836,
 'GEL': 2.631036,
 'GGP': 0.743836,
 'GHS': 11.568737,
 'GIP': 0.743836,
 'GMD': 74.331983,
 'GNF': 8777.194785,
 'GTQ': 7.626168,


In [62]:
# convert the rates dictionary to a DataFrame
exchange_rates = pd.DataFrame(list(rates.items()),columns=["currency", "exchange_rate"])
exchange_rates.head(10)  # Display the first few rows of the DataFrame

,currency,exchange_rate
0,USD,1.000000
1,AED,3.672500
2,AFN,66.008291
3,ALL,81.852701
4,AMD,366.674156
5,ANG,1.790000
6,AOA,926.855963
7,ARS,1480.122300
8,AUD,1.432241
9,AWG,1.790000


In [63]:
# Add base currency to every row
exchange_rates["base_currency"] = api_data["base_code"]

exchange_rates.head(10) 

,currency,exchange_rate,base_currency
0,USD,1.000000,USD
1,AED,3.672500,USD
2,AFN,66.008291,USD
3,ALL,81.852701,USD
4,AMD,366.674156,USD
5,ANG,1.790000,USD
6,AOA,926.855963,USD
7,ARS,1480.122300,USD
8,AUD,1.432241,USD
9,AWG,1.790000,USD


In [64]:
# Add extraction timestamp
exchange_rates["retrieved_at"] = api_data["time_last_update_utc"]

exchange_rates.head()

,currency,exchange_rate,base_currency,retrieved_at
0,USD,1.000000,USD,"Sat, 18 Jul 2026 00:02:32 +0000"
1,AED,3.672500,USD,"Sat, 18 Jul 2026 00:02:32 +0000"
2,AFN,66.008291,USD,"Sat, 18 Jul 2026 00:02:32 +0000"
3,ALL,81.852701,USD,"Sat, 18 Jul 2026 00:02:32 +0000"
4,AMD,366.674156,USD,"Sat, 18 Jul 2026 00:02:32 +0000"


In [65]:
exchange_rates

,currency,exchange_rate,base_currency,retrieved_at
0,USD,1.000000,USD,"Sat, 18 Jul 2026 00:02:32 +0000"
1,AED,3.672500,USD,"Sat, 18 Jul 2026 00:02:32 +0000"
2,AFN,66.008291,USD,"Sat, 18 Jul 2026 00:02:32 +0000"
3,ALL,81.852701,USD,"Sat, 18 Jul 2026 00:02:32 +0000"
4,AMD,366.674156,USD,"Sat, 18 Jul 2026 00:02:32 +0000"
...,...,...,...,...
161,YER,238.116844,USD,"Sat, 18 Jul 2026 00:02:32 +0000"
162,ZAR,16.502998,USD,"Sat, 18 Jul 2026 00:02:32 +0000"
163,ZMW,18.183446,USD,"Sat, 18 Jul 2026 00:02:32 +0000"
164,ZWG,26.684800,USD,"Sat, 18 Jul 2026 00:02:32 +0000"


In [67]:
# Rearrange the columns 
exchange_rates = exchange_rates[ 
    [               
        "base_currency",
        "currency",
        "exchange_rate",
        "retrieved_at"
    ]
]

exchange_rates.head()

,base_currency,currency,exchange_rate,retrieved_at
0,USD,USD,1.000000,"Sat, 18 Jul 2026 00:02:32 +0000"
1,USD,AED,3.672500,"Sat, 18 Jul 2026 00:02:32 +0000"
2,USD,AFN,66.008291,"Sat, 18 Jul 2026 00:02:32 +0000"
3,USD,ALL,81.852701,"Sat, 18 Jul 2026 00:02:32 +0000"
4,USD,AMD,366.674156,"Sat, 18 Jul 2026 00:02:32 +0000"


In [68]:
exchange_rates.shape

(166, 4)

In [69]:
exchange_rates.dtypes

base_currency        str
currency             str
exchange_rate    float64
retrieved_at         str
dtype: object